In [27]:
from client import *
from common_imports import *

In [28]:
# Add references
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

In [29]:
project_endpoint = MICROSOFT_FOUNDRY
model_deployment = AZURE_OPENAI_DEPLOYMENT_ID

In [30]:
# Connect to the agents client
with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    # Initialize agent MCP tool
    mcp_tool = MCPTool(
        server_label="api-specs",
        server_url="https://learn.microsoft.com/api/mcp",
        require_approval="always",
    )

    # Create a new agent with the MCP tool
    agent = project_client.agents.create_version(
        agent_name="MyAgent",
        definition=PromptAgentDefinition(
            model=model_deployment,
            instructions="You are a helpful agent that can use MCP tools to assist users. Use the available MCP tools to answer questions and perform tasks.",
            tools=[mcp_tool],
        ),
    )
    print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")
    # Create a conversation thread
    conversation = openai_client.conversations.create()
    print(f"Created conversation (id: {conversation.id})")

    response = openai_client.responses.create(
    conversation=conversation.id,
    input="Give me the Azure CLI commands to create an Azure Container App with a managed identity.",
   extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

    # Process any MCP approval requests that were generated
    # The agent may issue several tool calls, each needing its own approval,
    # so we loop until there are none left.
    while True:
        # Collect any MCP approval requests from the latest response
        input_list: ResponseInputParam = []
        for item in response.output:
            if item.type == "mcp_approval_request":
                if item.server_label == "api-specs" and item.id:
                    # Automatically approve the MCP request to allow the agent to proceed
                    input_list.append(
                        McpApprovalResponse(
                            type="mcp_approval_response",
                            approve=True,
                            approval_request_id=item.id,
                        )
                    )

        # No more approvals needed -> the agent has produced its final response
        if not input_list:
            break

        # Send the approval response back and retrieve the next response
        response = openai_client.responses.create(
            input=input_list,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

    print(f"\nAgent response: {response.output_text}")

Agent created (id: MyAgent:1, name: MyAgent, version: 1)

Created conversation (id: conv_f39494d6c44e854400WwUxU877QTkVYPPxArfmOcSoUe6Nfhgl)

Agent response: Here are Azure CLI commands for creating an Azure Container App with a managed identity:

### Example 1: With a System-Assigned Managed Identity
```bash
az containerapp create \
    --resource-group <RESOURCE_GROUP> \
    --name <APP_NAME> \
    --image <CONTAINER_IMAGE> \
    --environment <ENVIRONMENT_NAME> \
    --assign-identity system \
    --ingress 'external' \
    --target-port 8080 \
    --min-replicas 1
```
This will enable a system-assigned managed identity for your container app.

Learn more 
(https://learn.microsoft.com/azure/container-apps/tutorial-java-quarkus-connect-managed-identity-postgresql-databas
e#4-create-a-container-app-on-azure).

---

### Example 2: With a User-Assigned Managed Identity
```bash
az containerapp create \
    --resource-group <RESOURCE_GROUP> \
    --name <APP_NAME> \
    --environment <ENVIRONMENT_ID> \
    --user-assigned <USER_ASSIGNED_IDENTITY_ID> \
    --image <CONTAINER_IMAGE> \
    --registry-identity <USER_ASSIGNED_IDENTITY_ID> \
    --registry-server "<REGISTRY_NAME>.azurecr.io"
```
This command assigns a user-assigned managed identity for image registry access.

Learn more 
(https://learn.microsoft.com/azure/container-apps/managed-identity-image-pull?pivots=console#user-assigned-managed-
identity-1).

---

### Example 3: Enable Managed Identity for an Existing App
To assign a system-assigned identity to an existing app:
```bash
az containerapp identity assign \
    --name <APP_NAME> \
    --resource-group <RESOURCE_GROUP> \
    --system-assigned
```

Learn more (https://learn.microsoft.com/azure/container-apps/azure-pipelines#configuration).

Replace placeholders `<RESOURCE_GROUP>`, `<APP_NAME>`, `<CONTAINER_IMAGE>`, `<ENVIRONMENT_ID>`, and 
`<USER_ASSIGNED_IDENTITY_ID>` with your actual values.